# Question Answering

In [1]:
from datasets import Dataset, DatasetDict

train_data = {
    "question": [
        "Where does John work?",
        "Where does Alice live?",
        "Where does Bob work?",
        "Where does Emma live?",
        "Where does David work?",
        "Where does Sophia live?",
        "Where does Michael work?",
        "Where does Olivia live?"
    ],
    "context": [
        "John works at Google in California",
        "Alice lives in Paris and loves art",
        "Bob works at Microsoft in Seattle",
        "Emma lives in London and enjoys music",
        "David works at Amazon in New York",
        "Sophia lives in Berlin and studies design",
        "Michael works at Apple in Cupertino",
        "Olivia lives in Rome and likes history"
    ],
    "answers": [
        {"text": ["Google"], "answer_start": [14]},
        {"text": ["Paris"], "answer_start": [15]},
        {"text": ["Microsoft"], "answer_start": [13]},
        {"text": ["London"], "answer_start": [14]},
        {"text": ["Amazon"], "answer_start": [14]},
        {"text": ["Berlin"], "answer_start": [14]},
        {"text": ["Apple"], "answer_start": [16]},
        {"text": ["Rome"], "answer_start": [14]}
    ]
}

test_data = {
    "question": [
        "Where does Chris work?",
        "Where does Anna live?"
    ],
    "context": [
        "Chris works at Tesla in Texas",
        "Anna lives in Madrid and enjoys travel"
    ],
    "answers": [
        {"text": ["Tesla"], "answer_start": [14]},
        {"text": ["Madrid"], "answer_start": [13]}
    ]
}

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "test": Dataset.from_dict(test_data)
})


In [2]:
# 🔥 2. IMPORTS (same pattern)
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)
import numpy as np


In [3]:
# 🔥 3. MODEL + TOKENIZER

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [6]:

# 🔥 4. PREPROCESS (⚠️ MOST IMPORTANT DIFFERENCE)

# 👉 In QA, we must find:

# * **start position**
# * **end position**

def preprocess(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        truncation=True,
        padding="max_length",
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i in range(len(examples["answers"])):
        answer = examples["answers"][i]

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        offsets = inputs["offset_mapping"][i]
        sequence_ids = inputs.sequence_ids(i)

        start_token = 0
        end_token = 0

        for idx, (offset, seq_id) in enumerate(zip(offsets, sequence_ids)):
            if seq_id != 1:  # only context
                continue

            if offset[0] <= start_char < offset[1]:
                start_token = idx

            if offset[0] < end_char <= offset[1]:
                end_token = idx

        start_positions.append(start_token)
        end_positions.append(end_token)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    inputs.pop("offset_mapping")  # remove after use

    return inputs

dataset = dataset.map(preprocess, batched=True)


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [ ]:
# 🔥 5. TRAINING SETUP

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01
)


In [9]:

# 🔥 6. TRAINER

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    # tokenizer=tokenizer,
    data_collator=DefaultDataCollator()
)

# 🔥 7. TRAIN

trainer.train()

Epoch,Training Loss,Validation Loss
1,6.296638,6.276961
2,5.974495,6.113224
3,5.800227,6.025622


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=6.023786385854085, metrics={'train_runtime': 44.575, 'train_samples_per_second': 0.538, 'train_steps_per_second': 0.067, 'total_flos': 6271122161664.0, 'train_loss': 6.023786385854085, 'epoch': 3.0})

In [10]:
# 🔥 8. SAVE

trainer.save_model("qa_model")
tokenizer.save_pretrained("qa_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qa_model/tokenizer_config.json', 'qa_model/tokenizer.json')

In [13]:
# 🔥 9. INFERENCE

from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="qa_model",
    tokenizer="qa_model"
)

context = "Chris works at Tesla in Texas"
question = "Where does Chris work?"

result = qa_pipeline(
    question=question,
    context=context
)

print("result:", result)
print("Answer:", result["answer"])
print("Confidence:", round(result["score"], 2))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

result: {'score': 0.042513031512498856, 'start': 15, 'end': 20, 'answer': 'Tesla'}
Answer: Tesla
Confidence: 0.04


### Note

“The model is not generating the answer — it is selecting it from the context.”


### QA:

```
Question + Context → Answer span
```

---

# Important

> “QA model does NOT generate text — it **selects a span from the context**.”

---

# Internal Working (simple explanation)

Model predicts:

```
Start token → where answer begins
End token   → where answer ends
```

---

# ⚠️ Important Differences (Put these as comments in code)

```
# QA is different because:
# 1. We pass BOTH question and context
# 2. Labels are NOT classes → they are token positions
# 3. Model predicts start and end positions
# 4. Output is extracted from context (not generated)
```

---

# ⚠️ Reality Check

* Dataset is small → model may fail sometimes
* QA needs **lots of data normally**

---
